# NCF — Neural Collaborative Filtering (2017)

_Papers · Recommender Systems_

**Replace the dot-product in matrix factorization with a learned neural function, and fuse it with a multi-layer perceptron.**

---

This notebook is a practice scaffold for the **NCF — Neural Collaborative Filtering (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, numpy as np

torch.manual_seed(0); np.random.seed(0)

# --- 0. Sanity-check the worked example: GMF score = sigmoid(h^T (p_u (x) q_i)). ---
p_u = torch.tensor([0.5, -1.0, 2.0]); q_i = torch.tensor([1.0, 0.5, -0.5])
h   = torch.tensor([0.8, 0.4, -1.0])
elt = p_u * q_i                       # element-wise product (NOT the dot product)
z   = torch.dot(h, elt)               # h^T (p_u (x) q_i)
print("worked example:  p_u*q_i =", elt.tolist(), " z =", round(z.item(), 4),
      " yhat =", round(torch.sigmoid(z).item(), 4))
# worked example:  p_u*q_i = [0.5, -0.5, -1.0]  z = 1.2  yhat = 0.7685


# --- 1. A tiny implicit-feedback matrix with a NON-LINEAR preference pattern. ---
# Pure inner product (GMF) cannot represent the |p - q| "closeness" term; the MLP can.
NU, NI, K = 100, 120, 8
gu = torch.randn(NU, K); gi = torch.randn(NI, K)
lin  = 0.3 * (gu @ gi.t())
diff = (gu[:, :3].unsqueeze(1) - gi[:, :3].unsqueeze(0)).abs().sum(-1)  # |p - q| style
score = lin - 2.0 * diff + 1.0 * torch.randn(NU, NI)                    # + observation noise
pos = torch.zeros(NU, NI)
for u in range(NU):
    pos[u, score[u].topk(8).indices] = 1.0          # each user's top items = observed (label 1)
pos_pairs = [(u, i) for u in range(NU) for i in range(NI) if pos[u, i] == 1]

# Leave-one-out: hold out one true item per user for ranking.
loo = {}
for u in range(NU):
    its = (pos[u] == 1).nonzero().flatten().tolist()
    if len(its) >= 2:
        loo[u] = its[-1]; pos[u, its[-1]] = 0       # remove from training

NEG = 4                                              # negatives sampled per positive
def make_dataset():                                  # resampled each epoch (sec 3.1)
    us, it, lb = [], [], []
    for (u, i) in pos_pairs:
        us += [u]; it += [i]; lb += [1.0]
        for _ in range(NEG):
            j = np.random.randint(NI)
            while pos[u, j] == 1: j = np.random.randint(NI)
            us += [u]; it += [j]; lb += [0.0]
    return torch.tensor(us), torch.tensor(it), torch.tensor(lb)


# --- 2. The two models (built by hand). ---
class GMF(nn.Module):                                # learned inner product ~ matrix factorization
    def __init__(self):
        super().__init__()
        self.Pu = nn.Embedding(NU, K); self.Qi = nn.Embedding(NI, K)
        self.h  = nn.Linear(K, 1)                    # the weight vector h (all-ones => plain MF)
    def forward(self, u, i):
        return self.h(self.Pu(u) * self.Qi(i)).squeeze(-1)   # h^T (p_u (x) q_i)  [raw logit]

class NeuMF(nn.Module):                              # GMF branch fused with an MLP tower (sec 3.4)
    def __init__(self, mf_dim=K, mlp_dim=16):
        super().__init__()
        self.PuG = nn.Embedding(NU, mf_dim);  self.QiG = nn.Embedding(NI, mf_dim)   # GMF embeddings
        self.PuM = nn.Embedding(NU, mlp_dim); self.QiM = nn.Embedding(NI, mlp_dim)  # separate MLP embeddings
        self.mlp = nn.Sequential(                    # tower over concatenated embeddings
            nn.Linear(2 * mlp_dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 8),  nn.ReLU())
        self.h = nn.Linear(mf_dim + 8, 1)            # final weight over [phi_GMF ; phi_MLP]
    def forward(self, u, i):
        gmf = self.PuG(u) * self.QiG(i)                              # phi_GMF
        mlp = self.mlp(torch.cat([self.PuM(u), self.QiM(i)], -1))    # phi_MLP
        return self.h(torch.cat([gmf, mlp], -1)).squeeze(-1)        # sigma applied by the loss


def train(model, epochs=60, bs=128):
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-6)
    bce = nn.BCEWithLogitsLoss()                     # binary cross-entropy (sigmoid built in)
    for ep in range(epochs):
        u, i, y = make_dataset()
        perm = torch.randperm(len(y)); u, i, y = u[perm], i[perm], y[perm]
        model.train()
        for b in range(0, len(y), bs):
            opt.zero_grad()
            loss = bce(model(u[b:b+bs], i[b:b+bs]), y[b:b+bs]); loss.backward(); opt.step()
    return model


# --- 3. Leave-one-out ranking: HR@10 and NDCG@10 (1 true item vs 99 sampled negatives). ---
def evaluate(model, K_rank=10, n_neg=99):
    model.eval(); hrs, ndcgs = [], []
    with torch.no_grad():
        for u, true_i in loo.items():
            negs = []
            while len(negs) < n_neg:
                j = np.random.randint(NI)
                if pos[u, j] == 0 and j != true_i: negs.append(j)
            cand = torch.tensor([true_i] + negs)
            uu = torch.full((len(cand),), u, dtype=torch.long)
            order = model(uu, cand).argsort(descending=True)
            rank = (order == 0).nonzero().item()     # 0-indexed rank of the true item
            hrs.append(1.0 if rank < K_rank else 0.0)
            ndcgs.append((1.0 / np.log2(rank + 2)) if rank < K_rank else 0.0)
    return float(np.mean(hrs)), float(np.mean(ndcgs))


torch.manual_seed(1); np.random.seed(1)
hr_g, ndcg_g = evaluate(train(GMF()))
torch.manual_seed(1); np.random.seed(2)
hr_n, ndcg_n = evaluate(train(NeuMF()))

print("\nLeave-one-out ranking (1 positive vs 99 sampled negatives, @10):")
print(f"  GMF   (ablation):  HR@10 = {hr_g:.3f}   NDCG@10 = {ndcg_g:.3f}")
print(f"  NeuMF (GMF + MLP): HR@10 = {hr_n:.3f}   NDCG@10 = {ndcg_n:.3f}")
# our small run, not the paper's numbers:
#   GMF   : HR@10 = 0.970   NDCG@10 = 0.661
#   NeuMF : HR@10 = 1.000   NDCG@10 = 0.981
# NeuMF beats plain GMF on both, with the larger gap on NDCG (it ranks the true item higher).

## Visualize the data & results

_On held-out ranking, does fusing GMF with an MLP (NeuMF) beat plain GMF (the dot-product generalization, ~matrix factorization)?_

In [ ]:
import torch, torch.nn as nn, numpy as np

# Same toy setup as CODE; evaluate NDCG@10 every 5 epochs to draw a learning curve.
NU, NI, K = 100, 120, 8
torch.manual_seed(0); np.random.seed(0)
gu = torch.randn(NU, K); gi = torch.randn(NI, K)
score = 0.3 * (gu @ gi.t()) \
        - 2.0 * (gu[:, :3].unsqueeze(1) - gi[:, :3].unsqueeze(0)).abs().sum(-1) \
        + 1.0 * torch.randn(NU, NI)
pos = torch.zeros(NU, NI)
for u in range(NU): pos[u, score[u].topk(8).indices] = 1.0
pos_pairs = [(u, i) for u in range(NU) for i in range(NI) if pos[u, i] == 1]
loo = {}
for u in range(NU):
    its = (pos[u] == 1).nonzero().flatten().tolist()
    if len(its) >= 2: loo[u] = its[-1]; pos[u, its[-1]] = 0

def make_dataset(NEG=4):
    us, it, lb = [], [], []
    for (u, i) in pos_pairs:
        us += [u]; it += [i]; lb += [1.0]
        for _ in range(NEG):
            j = np.random.randint(NI)
            while pos[u, j] == 1: j = np.random.randint(NI)
            us += [u]; it += [j]; lb += [0.0]
    return torch.tensor(us), torch.tensor(it), torch.tensor(lb)

class GMF(nn.Module):
    def __init__(s):
        super().__init__(); s.Pu=nn.Embedding(NU,K); s.Qi=nn.Embedding(NI,K); s.h=nn.Linear(K,1)
    def forward(s,u,i): return s.h(s.Pu(u)*s.Qi(i)).squeeze(-1)
class NeuMF(nn.Module):
    def __init__(s, mf=K, md=16):
        super().__init__()
        s.PuG=nn.Embedding(NU,mf); s.QiG=nn.Embedding(NI,mf)
        s.PuM=nn.Embedding(NU,md); s.QiM=nn.Embedding(NI,md)
        s.mlp=nn.Sequential(nn.Linear(2*md,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,8),nn.ReLU())
        s.h=nn.Linear(mf+8,1)
    def forward(s,u,i):
        return s.h(torch.cat([s.PuG(u)*s.QiG(i), s.mlp(torch.cat([s.PuM(u),s.QiM(i)],-1))],-1)).squeeze(-1)

def ndcg(model, Kr=10, nn_=99):
    model.eval(); v=[]
    with torch.no_grad():
        for u,ti in loo.items():
            negs=[]
            while len(negs)<nn_:
                j=np.random.randint(NI)
                if pos[u,j]==0 and j!=ti: negs.append(j)
            cand=torch.tensor([ti]+negs); uu=torch.full((len(cand),),u)
            r=(model(uu,cand).argsort(descending=True)==0).nonzero().item()
            v.append((1.0/np.log2(r+2)) if r<Kr else 0.0)
    return float(np.mean(v))

def run(model, seed, epochs=60, bs=128):
    torch.manual_seed(seed); np.random.seed(seed)
    opt=torch.optim.Adam(model.parameters(),lr=0.01,weight_decay=1e-6); bce=nn.BCEWithLogitsLoss()
    curve=[]
    for ep in range(epochs):
        u,i,y=make_dataset(); p=torch.randperm(len(y)); u,i,y=u[p],i[p],y[p]
        model.train()
        for b in range(0,len(y),bs):
            opt.zero_grad(); bce(model(u[b:b+bs],i[b:b+bs]),y[b:b+bs]).backward(); opt.step()
        if ep%5==0 or ep==epochs-1: curve.append([ep, round(ndcg(model),3)])
    return curve

torch.manual_seed(1)
print("GMF  NDCG@10:", run(GMF(), 1))
print("NeuMF NDCG@10:", run(NeuMF(), 2))
# GMF   -> plateaus ~0.71 ; NeuMF -> climbs to ~0.99 (our small run, not the paper's numbers).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working NeuMF that ranks well. Delete the entire MLP branch (and
            its embeddings), leaving plain GMF &mdash; a learned inner product, essentially matrix
            factorization. Retrain on the same data, keeping everything else identical, and re-measure HR@10
            and NDCG@10. What happens, and what does it demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Remove only the MLP branch: drop PuM, QiM, self.mlp and fuse nothing &mdash; score from h(p_u &odot; q_i) alone. Keep $K$, optimizer, negatives, epochs, and seeds the same. — _An honest ablation changes exactly one thing &mdash; the non-linear MLP tower &mdash; so any drop is attributable to it._
- Retrain and compare ranking: GMF's HR@10 and especially NDCG@10 fall below NeuMF's. — _If the data hides a non-linear user&ndash;item pattern, the dot-product generalization cannot represent it; the MLP can, so removing it loses ranking accuracy._
- Note the gap is larger on NDCG@10 than on HR@10. — _HR only asks "in the top 10?"; NDCG also rewards ranking the true item higher, where the extra non-linear signal helps most._

**Answer:** GMF ranks worse than NeuMF &mdash; in our small run NDCG@10 drops from about $0.98$ to about
                 $0.66$ and HR@10 from $1.00$ to about $0.97$ (our small run, not the paper's numbers). Since
                 the only change was deleting the MLP branch, this isolates the learned non-linear interaction
                 as the source of NeuMF's advantage: fusing GMF with the MLP beats the dot-product
                 generalization alone, reproducing the paper's qualitative effect.

</details>

**Problem 2.** Recompute the worked example. With $p_u=[0.5,-1.0,2.0]$, $q_i=[1.0,0.5,-0.5]$, and
            $h=[0.8,0.4,-1.0]$, what is the GMF score $\hat{y}_{ui}$? Then state what plain matrix
            factorization (the all-ones $h$) would have scored, and what that difference shows.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Element-wise product: $p_u \odot q_i = [0.5,\,-0.5,\,-1.0]$. — _Multiply matching entries; keep the vector (do not sum yet)._
- Weighted sum with $h$: $z = 0.8(0.5)+0.4(-0.5)+(-1.0)(-1.0) = 1.2$. — _$h^\top(p_u\odot q_i)$ &mdash; the learned weights re-scale each factor._
- Sigmoid: $\hat{y}_{ui}=\sigma(1.2)\approx 0.769$. All-ones $h$ gives inner product $-1.0$, so $\sigma(-1.0)\approx 0.269$. — _Same embeddings, different combiner: the learned $h$ flips a low score into a high one._

**Answer:** GMF scores $\sigma(1.2)\approx 0.769$. Plain matrix factorization (the all-ones $h$) would
                 score the inner product $-1.0$ &rarr; $\sigma(-1.0)\approx 0.269$. The learned weight vector
                 $h$ re-weighted the latent factors and turned a "no" into a "yes" &mdash; the extra
                 flexibility that makes GMF a generalization of matrix factorization.

</details>

**Problem 3.** A teammate trains NeuMF on implicit data using only the observed (label $1$) interactions, no
            negatives, and reports a near-zero binary-cross-entropy loss but terrible recommendations. What
            went wrong, and how do you fix both the training and the evaluation?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Diagnose training: with only label-$1$ examples, predicting $\hat{y}=1$ everywhere drives the loss to ~0 while learning nothing about what users dislike. — _Binary cross-entropy needs both classes; all-positive data has a trivial degenerate optimum._
- Fix training: add negative sampling &mdash; for each positive, draw a few unobserved $(u,j)$ pairs as label $0$, resampled each epoch (&sect;3.1). — _Negatives teach the model to rank observed items above unobserved ones._
- Fix evaluation: stop trusting the loss; use leave-one-out HR@10 / NDCG@10 ranking instead. — _Recommendation quality is about ordering items, which a scalar loss does not directly measure._

**Answer:** Training on positives only lets the model output $1$ for everything &mdash; loss near zero,
                 recommendations useless. Add negative sampling (unobserved pairs as label $0$, resampled each
                 epoch) so the model learns to separate liked from not-liked, and judge it with a ranking
                 metric (leave-one-out HR@10 / NDCG@10), not the loss value.

</details>